In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
product_data = pd.read_excel("/content/sample_data/Test Data 2.xlsx")
category_data=pd.read_excel("/content/sample_data/full unspsc list.xlsx")

In [ ]:
category_data.head()

In [ ]:
product_data.dropna(subset=['Short Text'],inplace=True)

In [ ]:
nlp = spacy.load("en_core_web_lg")

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
def filter_data(text):
    nouns=[]
    doc=nlp(text)
    for token in doc:
        if token.pos_ in ["NOUN","PROPN"]:
            nouns.append(token.text)
    return " ".join(nouns)

In [ ]:
product_data['With Nouns']= product_data['Short Text'].apply(filter_data)

In [ ]:
def clean_text(text):
    text=text.lower()
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

In [ ]:
tfidf=TfidfVectorizer(binary=True,use_idf=False, norm=None)

In [ ]:
tfidf.fit(category_data['Level 4 Description'])

In [ ]:
product_vectors = tfidf.transform(product_data['With Nouns'])
category_vectors = tfidf.transform(category_data['Level 4 Description'])

In [ ]:
similarity_matrix = cosine_similarity(product_vectors, category_vectors)

In [ ]:
similarity_matrix.shape

In [ ]:
model= SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
refined_mapped_categories = []

for i in range(len(product_data)):
    similarity_scores = similarity_matrix[i]
    top_indices = similarity_scores.argsort()[-100:][::-1]

    top_100_categories= category_data['Level 4 Description'].iloc[top_indices].tolist()

    product_embedding = model.encode(product_data['With Nouns'][i])
    category_embeddings = model.encode(top_100_categories)

    refined_similarity=model.similarity(product_embedding,category_embeddings)[0]
    refined_similarity_array=np.array(refined_similarity)

    top_5_indices = refined_similarity_array.argsort()[-5:][::-1]

#     print(top_5_indices, type(top_5_indices))

    top_5_categories=[]
    for idx in top_5_indices:
        category = top_100_categories[idx]
        top_5_categories.append(category)

    refined_mapped_categories.append(", ".join(top_5_categories))
    break

# product_data['Mapped Category'] = refined_mapped_categories
print(refined_mapped_categories)

In [ ]:
product_data['Short Text'][0]

In [ ]:
refined_mapped_categories = []

for i in range(len(product_data)):
    similarity_scores = similarity_matrix[i]
    top_indices = similarity_scores.argsort()[-100:][::-1]

    top_100_categories= category_data['Level 4 Description'].iloc[top_indices].tolist()

    product_embedding = model.encode(product_data['With Nouns'][i])
    category_embeddings = model.encode(top_100_categories)

    refined_similarity=model.similarity(product_embedding,category_embeddings)[0]
    refined_similarity_array=np.array(refined_similarity)

    top_5_indices = refined_similarity_array.argsort()[-5:][::-1]

#     print(top_5_indices, type(top_5_indices))

    top_5_categories=[]
    for idx in top_5_indices:
        category = top_100_categories[idx]
        top_5_categories.append(category)

    refined_mapped_categories.append(", ".join(top_5_categories))

product_data['Mapped Category'] = refined_mapped_categories

In [ ]:
category_data = category_data.reset_index(drop=True)

refined_mapped_categories = []

for i in range(len(product_data)):
    similarity_scores = similarity_matrix[i]
    top_indices = similarity_scores.argsort()[-100:][::-1]

    top_100_categories = category_data['Level 4 Description'].iloc[top_indices].tolist()

    product_embedding = model.encode(product_data['With Nouns'][i])
    category_embeddings = model.encode(top_100_categories)

    refined_similarity = model.similarity(product_embedding, category_embeddings)[0]
    refined_similarity_array = np.array(refined_similarity)

    top_5_indices = refined_similarity_array.argsort()[-5:][::-1]

    top_5_categories = [top_100_categories[idx] for idx in top_5_indices]

    refined_mapped_categories.append(", ".join(top_5_categories))

product_data['Mapped Category'] = refined_mapped_categories

In [ ]:
refined_mapped_categories = []

for index, row in product_data.iterrows():
    similarity_scores = similarity_matrix[index]
    top_indices = similarity_scores.argsort()[-100:][::-1]

    top_100_categories= category_data['Level 4 Description'].iloc[top_indices].tolist()

    product_embedding = model.encode(row['With Nouns'])
    category_embeddings = model.encode(top_100_categories)

    refined_similarity=model.similarity(product_embedding,category_embeddings)[0]
    refined_similarity_array=np.array(refined_similarity)

    top_5_indices = refined_similarity_array.argsort()[-5:][::-1]

    top_5_categories=[]
    for idx in top_5_indices:
        category = top_100_categories[idx]
        top_5_categories.append(category)

    refined_mapped_categories.append(", ".join(top_5_categories))

product_data['Mapped Category'] = refined_mapped_categories

In [ ]:
refined_mapped_categories = []

for index, row in product_data.iterrows():
    similarity_scores = similarity_matrix[index if index < len(similarity_matrix) else len(similarity_matrix) - 1]
    top_indices = similarity_scores.argsort()[-100:][::-1]

    top_100_categories= category_data['Level 4 Description'].iloc[top_indices].tolist()

    product_embedding = model.encode(row['With Nouns'])
    category_embeddings = model.encode(top_100_categories)

    refined_similarity=model.similarity(product_embedding,category_embeddings)[0]
    refined_similarity_array=np.array(refined_similarity)

    top_5_indices = refined_similarity_array.argsort()[-5:][::-1]

    top_5_categories=[]
    for idx in top_5_indices:
        category = top_100_categories[idx]
        top_5_categories.append(category)

    refined_mapped_categories.append(", ".join(top_5_categories))

product_data['Mapped Category'] = refined_mapped_categories

In [ ]:
product_data.to_excel('TFIDF_Transformers.xlsx', index=False)

In [ ]:
product_data['Mapped Category'][18012]